In [22]:
import os
import csv
import html
import re
from datetime import datetime

BASE_DIR = "complaints"

def safe_slug(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"\s+", "_", text)
    return text or "none"

def get_youtube_embed_url(url):
    clean_url = url.replace("\\n", "").strip()
    match = re.search(r'(?:v=|\/)([0-9A-Za-z_-]{11}).*', clean_url)
    if match:
        vid_id = match.group(1)
        if len(vid_id) == 11:
            return f"https://www.youtube.com/embed/{vid_id}"
    return None

def generate_html_for_unit(unit_dir, address, unit_name):
    csv_path = os.path.join(unit_dir, "complaints.csv")
    if not os.path.exists(csv_path):
        return

    complaints = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            complaints.append(row)

    # Gather all image files in the unit directory
    all_images = [f for f in os.listdir(unit_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.gif'))]
    used_images = set()

    # Build HTML content
    html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Complaints Dashboard - {html.escape(address)}</title>
    <script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-gray-50 text-gray-800 font-sans min-h-screen p-4 sm:p-8">
    <div class="max-w-4xl mx-auto">
        
        <!-- Header -->
        <header class="bg-white shadow rounded-lg p-6 mb-6 border border-gray-200">
            <h1 class="text-2xl font-bold text-gray-900">{html.escape(address)}</h1>
            <p class="text-sm text-gray-500 mt-1">Unit / Identifier: <span class="font-semibold text-gray-700">{html.escape(unit_name)}</span></p>
            <div class="mt-3 flex items-center gap-4 text-sm">
                <span class="bg-indigo-50 text-indigo-700 font-medium px-3 py-1 rounded-full border border-indigo-100">
                    Total Complaints: {len(complaints)}
                </span>
                <span class="bg-gray-100 text-gray-700 font-medium px-3 py-1 rounded-full border border-gray-200">
                    Total Evidence Files: {len(all_images)}
                </span>
            </div>
        </header>
        
        <!-- Complaints List -->
        <div class="space-y-6">
"""

    for c in complaints:
        timestamp = html.escape(c.get("Timestamp", ""))
        nature_raw = c.get("Nature of Complaint", "unknown")
        nature = html.escape(nature_raw)
        desc = html.escape(c.get("Detailed Description of Incident", ""))
        date_inc = c.get("Date of Incident", "").strip()
        time_inc = html.escape(c.get("Time of Incident (Optional)", ""))
        impact = html.escape(c.get("How would you rate the impact of this incident on your quality of life?", ""))
        
        # Determine date slug for image matching
        try:
            parsed_date = datetime.strptime(date_inc, "%m/%d/%Y")
            date_slug = parsed_date.strftime("%Y-%m-%d")
        except:
            date_slug = "unknown_date"

        nature_slug = safe_slug(nature_raw)
        img_prefix = f"{nature_slug}_{date_slug}"

        # Find images specific to this complaint
        complaint_images = []
        for img in all_images:
            if img not in used_images and img.startswith(img_prefix):
                complaint_images.append(img)
                used_images.add(img)

        links_raw = c.get("Links (YouTube or other URLs - Please provide one per line)", "")
        links = [l.strip() for l in links_raw.replace("\\n", "\n").split("\n") if l.strip()]

        html_content += f"""
            <div class="bg-white shadow rounded-lg p-6 border border-gray-200 space-y-4">
                <div class="flex flex-wrap justify-between items-start gap-2 border-b pb-4">
                    <div>
                        <span class="bg-red-100 text-red-800 text-xs font-semibold px-2.5 py-1 rounded-md">{nature}</span>
                        <p class="text-sm text-gray-500 mt-2">Incident Date: <span class="font-medium text-gray-700">{html.escape(date_inc)} {time_inc}</span></p>
                    </div>
                    <div class="text-right">
                        <span class="text-xs text-gray-400 block">Logged: {timestamp}</span>
                        <span class="inline-block bg-amber-50 text-amber-800 text-xs font-semibold px-2.5 py-1 rounded-md mt-1 border border-amber-100">
                            Impact Rating: {impact}/5
                        </span>
                    </div>
                </div>

                <div>
                    <h3 class="text-xs font-bold text-gray-400 uppercase tracking-wider mb-1">Description</h3>
                    <p class="text-gray-700 whitespace-pre-line leading-relaxed">{desc}</p>
                </div>
"""

        # YouTube Embeds & Links
        if links:
            html_content += """
                <div>
                    <h3 class="text-xs font-bold text-gray-400 uppercase tracking-wider mb-2">Attached Links & Videos</h3>
                    <div class="space-y-3">
"""
            for link in links:
                safe_link = html.escape(link)
                embed_url = get_youtube_embed_url(link)
                if embed_url:
                    html_content += f"""
                        <div class="my-2">
                            <div class="relative w-full aspect-video rounded-lg overflow-hidden bg-black shadow-inner border border-gray-200 max-w-xl">
                                <iframe src="{embed_url}" class="absolute top-0 left-0 w-full h-full border-0" allowfullscreen></iframe>
                            </div>
                            <a href="{safe_link}" target="_blank" class="text-xs text-indigo-600 hover:underline inline-block mt-1">📺 Watch on YouTube ({safe_link})</a>
                        </div>
                    """
                else:
                    html_content += f'<div><a href="{safe_link}" target="_blank" class="text-indigo-600 hover:underline text-sm font-medium">🔗 {safe_link}</a></div>'
            html_content += """
                    </div>
                </div>
"""

        # Complaint-Specific Image Gallery
        if complaint_images:
            html_content += """
                <div class="pt-3 border-t border-gray-100">
                    <h3 class="text-xs font-bold text-gray-400 uppercase tracking-wider mb-3">Evidence Photos for this Incident</h3>
                    <div class="grid grid-cols-2 sm:grid-cols-3 gap-3">
"""
            for img in complaint_images:
                safe_img = html.escape(img)
                html_content += f"""
                        <a href="{safe_img}" target="_blank" class="block group">
                            <div class="aspect-square bg-gray-100 rounded-lg overflow-hidden border border-gray-200 shadow-sm">
                                <img src="{safe_img}" alt="Evidence Photo" class="w-full h-full object-cover group-hover:scale-105 transition duration-200">
                            </div>
                            <p class="text-xs text-gray-500 mt-1 truncate">{safe_img}</p>
                        </a>
                """
            html_content += """
                    </div>
                </div>
"""

        html_content += """
            </div>
"""

    html_content += """
        </div>
    </div>
</body>
</html>
"""

    index_path = os.path.join(unit_dir, "index.html")
    with open(index_path, "w", encoding="utf-8") as f:
        f.write(html_content)
    print(f"[+] Generated enhanced dashboard → {index_path}")


def crawl_and_generate():
    if not os.path.exists(BASE_DIR):
        print(f"[-] Directory '{BASE_DIR}' does not exist.")
        return

    print("[+] Scanning directories to generate grouped & embedded dashboards...")
    count = 0
    for root, dirs, files in os.walk(BASE_DIR):
        if "complaints.csv" in files:
            unit_dir = root
            parent_dir = os.path.dirname(unit_dir)
            address_slug = os.path.basename(parent_dir)
            unit_name = os.path.basename(unit_dir)
            
            address = address_slug.replace("_", " ").title()
            
            csv_path = os.path.join(unit_dir, "complaints.csv")
            try:
                with open(csv_path, "r", encoding="utf-8") as f:
                    reader = csv.DictReader(f)
                    first_row = next(reader, None)
                    if first_row and first_row.get("Address of Rental Property"):
                        address = first_row.get("Address of Rental Property")
            except:
                pass

            generate_html_for_unit(unit_dir, address, unit_name)
            count += 1

    print(f"[+] Complete! Generated {count} dashboard(s).")


if __name__ == "__main__":
    crawl_and_generate()

[+] Scanning directories to generate grouped & embedded dashboards...
[+] Generated enhanced dashboard → complaints\8611_whitefield_ave_savannah_ga_31406\svr-12345_placeholder\index.html
[+] Generated enhanced dashboard → complaints\9240_garland_dr_savannah_ga_31406\none\index.html
[+] Complete! Generated 2 dashboard(s).
